# 05 - Avaliação final no teste de 2015

Bloco 8 do plano da Parte 2. É o único sítio onde o teste de 2015 é usado. Os modelos chegam aqui **congelados**: a configuração de cada um, o limiar da classificação e o pipeline treinado em todo o treino foram gravados pelo `scripts/freeze_models.py` antes de o teste ser aberto, e o JSON de cada tarefa está no Git com a tag `modelos-congelados`.

**Regras deste notebook**

- Não treina nem escolhe nada: lê os pipelines congelados (confirmando o hash de cada ficheiro) e prevê o teste.
- As previsões ficam guardadas em `data/processed/test_predictions/`. Voltar a correr o notebook lê-as do ficheiro; se as escolhas congeladas mudarem, o notebook para com um erro.
- Se um resultado do teste for mau, o modelo **não se muda**: discute-se no relatório.

**Antes de correr:** congelar as duas tarefas, fazer commit dos dois JSON e a tag (ver `docs/05_avaliacao/avaliacao_final.md`).

| Requisito do enunciado | Onde |
|---|---|
| *"Preserve a final test set for final evaluation"* | Secção 1 (tag e hashes) e todo o notebook |
| *"The mean and the variation of the metric in the cross-validation procedures and the mean in the test dataset"* | Secções 3.2 e 4.2 |
| *"One comparison table for each task, including baselines"* | Secções 3.2 e 4.2 |
| Matriz de confusão, *precision*, *recall* e F1 (macro e micro), AUC, *accuracy* | Secções 4.2 e 4.4 |
| Incerteza no teste (bootstrap, T03) | Secções 3.2, 3.3, 4.2 e 4.3 |
| *"Selection bias, unequal follow-up, temporal change, fairness"* | Secções 2, 3.4, 3.5, 4.5, 4.6 e 5 |

In [ ]:
# Setup: imports e caminhos do projeto
import sys
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lending import config
from lending.cohorts import TARGET_CLASSES
from lending.experiment import RESULTS_DIR
from lending.final_evaluation import (TAG, FinalEvaluation, load_frozen, parse_mean_std,
                                      tag_status, test_cohort)
from lending.plot_style import apply_style
from lending.storage import PickleStore

apply_style()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REQUIRE_TAG = True    # False só para ensaiar o notebook com dados falsos
N_BOOT = 1000         # reamostragens do bootstrap (T03, p. 41 e 62)
GROUPS = ["região", "home_ownership", "rendimento"]
TASKS = ["regression", "classification"]

## 1. Os modelos congelados

Para cada tarefa: o JSON congelado tem de ser igual ao que está na tag `modelos-congelados` (senão, alguma escolha mudou depois da tag). A tabela mostra o que vai ao teste: a melhor configuração de cada modelo na validação cruzada, as *baselines* e o modelo final.

In [ ]:
frozen = {task: load_frozen(task) for task in TASKS}
status = {task: tag_status(task) for task in TASKS}
for task in TASKS:
    record = frozen[task]
    print(f"{task}: tag {status[task]} | congelado em {record['frozen_at']} (commit {record['git_commit']}) | "
          f"modelo final: {record['final_model']}")
    if record.get("uncommitted_code_changes"):
        print(f"  AVISO: havia código por gravar quando {task} foi congelada")
problems = {task: s for task, s in status.items() if s != "ok"}
if problems and REQUIRE_TAG:
    raise RuntimeError(f"Sem a tag {TAG} certa: {problems}. Ver docs/05_avaliacao/avaliacao_final.md")

rows = []
for task in TASKS:
    metric = "mae" if task == "regression" else "f1_macro"
    for e in frozen[task]["models"]:
        rows.append({"tarefa": task, "modelo": e["model"], "final": "sim" if e["model"] == frozen[task]["final_model"] else "",
                     "configuração": e["params"] or "—", "limiar": e["threshold"],
                     "métrica da escolha (CV)": e["cv"].get(metric, "—"), "n treino": e["n_train"],
                     "treino (s)": e["fit_seconds"], "ficheiro (MB)": e["size_mb"]})
pd.DataFrame(rows).set_index(["tarefa", "modelo"])

## 2. Coortes de teste

As coortes de 2015 saem do mesmo `CohortBuilder` do treino, com `partition="test"`, a partir do teste limpo na Parte 1 com as mesmas regras (`docs/02_preprocessamento/teste_final.md`). Todas as linhas removidas ficam registadas.

- **Regressão:** todos os empréstimos de 2015 com `int_rate` (36 e 60 meses, incluindo os ainda ativos: a taxa é conhecida no pedido).
- **Classificação:** só 36 meses e só com o desfecho conhecido. Os empréstimos de 2015 a 36 meses acabaram o prazo até ao fim de 2018, antes da extração (2019-03), por isso quase todos têm desfecho. Os *At risk* (ainda em atraso depois do fim do prazo) contam como *Default*, como decidido na Parte 1; a coluna `target_3` guarda as 3 classes do enunciado para a tabela 3 × 2.

In [ ]:
raw_test = PickleStore.load(config.TEST_PKL, expected_type=pd.DataFrame)
test_reg, log_reg = test_cohort("regression", raw_test)
test_cls, log_cls = test_cohort("classification", raw_test)
print(f"Teste limpo: {len(raw_test):,} empréstimos | regressão: {len(test_reg):,} | classificação: {len(test_cls):,}")
pd.concat({"regressão": log_reg.to_frame(), "classificação": log_cls.to_frame()})

In [ ]:
# Classes da classificação: as 3 do enunciado e as 2 usadas, no treino (2007-2011) e no teste (2015)
train_cls = PickleStore.load(config.CLS_TRAIN_PKL, expected_type=pd.DataFrame)
classes = pd.DataFrame({
    "treino": train_cls["loan_status"].map(TARGET_CLASSES).value_counts(),
    "teste": test_cls["target_3"].value_counts()}).reindex(["Good standing", "At risk", "Default"]).fillna(0).astype(int)
classes["teste (%)"] = (100 * classes["teste"] / classes["teste"].sum()).round(2)
classes["teste, binário"] = ["Good standing", "Default", "Default"]
classes

## 3. Regressão: taxa de juro em T0

### 3.1 Previsões

Cada pipeline congelado (pré-processamento + modelo, treinado em todo o treino de 2013–2014) prevê o teste uma vez. O tempo de treino é o do congelamento; o de previsão é o deste notebook (ou o da primeira execução, se as previsões já estavam guardadas).

In [ ]:
train_reg = PickleStore.load(config.REG_DEV_PKL, expected_type=pd.DataFrame)
reg = FinalEvaluation("regression", train_reg, test_reg, frozen=frozen["regression"])
reg.predict()
reg.timing

### 3.2 Tabela de comparação: validação cruzada e teste

Uma linha por modelo, pela ordem da validação cruzada (a escolha não muda com o teste). "CV" é a média ± desvio-padrão nos 6 folds trimestrais; "teste" é o valor em 2015, com o intervalo de confiança a 95% por bootstrap (1 000 reamostragens dos empréstimos de teste).

**Como ler:** um MAE de teste dentro da variação entre folds quer dizer que o modelo generaliza para um ano que nunca viu. Um MAE claramente acima é sinal de mudança temporal (secção 3.4), não de *overfitting* à validação: as escolhas foram feitas antes de ver 2015.

In [ ]:
reg_table = reg.metrics_table(N_BOOT)
reg_table.to_csv(RESULTS_DIR / "regression_test.csv")
reg_table

### 3.3 O modelo final contra os outros (bootstrap emparelhado)

A diferença de MAE entre o modelo final e cada um dos outros, nas mesmas reamostragens do teste. Negativo quer dizer que o modelo final erra menos. Se o intervalo contém o 0, os dois não se distinguem no teste.

In [ ]:
reg_paired = reg.paired_table(n_boot=N_BOOT)
reg_paired.to_csv(RESULTS_DIR / "regression_test_paired.csv")
reg_paired

In [ ]:
ax = reg.plot_cv_vs_test("mae")
ax.set_title("Regressão: MAE na validação cruzada e no teste de 2015")
ax.figure.tight_layout()
ax.figure.savefig(config.FIGURES_DIR / "final_reg_cv_teste.png")
plt.show()

### 3.4 Mudança temporal: o erro ao longo de 2015

O 04b mostrou o MAE de cada trimestre de validação em 2013–2014. Aqui continua-se a mesma linha em 2015, mês a mês, com o modelo final. Um viés sistemático (sempre acima ou abaixo de 0) quer dizer que o nível das taxas mudou face ao treino; um MAE a subir ao longo do ano quer dizer que o modelo envelhece e precisa de ser retreinado (plano de deployment).

In [ ]:
reg_months = reg.by_period()
axes = reg.plot_by_period()
axes[0].figure.suptitle(f"Regressão ({reg.final_model}): erro por mês de emissão em 2015")
axes[0].figure.tight_layout()
axes[0].figure.savefig(config.FIGURES_DIR / "final_reg_por_mes.png")
plt.show()
reg_months

### 3.5 Equidade: o erro por grupo

O enunciado pede para discutir a equidade. A geografia ficou fora dos modelos (possível *proxy* de grupos protegidos), por isso mede-se aqui se o erro é igual nas regiões dos EUA, nos tipos de habitação e nos quintis de rendimento (com os limites do treino). Um viés positivo num grupo quer dizer que o modelo lhe "cobra" mais do que a LendingClub cobrou. Grupos com menos de 1 000 empréstimos juntam-se em "outros".

In [ ]:
reg_groups = pd.concat({by: reg.subgroup_table(by) for by in GROUPS})
reg_groups.to_csv(RESULTS_DIR / "regression_test_subgroups.csv")
fig = reg.plot_subgroups(GROUPS, ["MAE", "viés"])
fig.savefig(config.FIGURES_DIR / "final_reg_equidade.png")
plt.show()
reg_groups

## 4. Classificação: risco de *default* em T4

### 4.1 Previsões

Cada pipeline congelado (treinado em todos os empréstimos de 36 meses de 2007–2011) dá a P(Default) de cada empréstimo de teste. A classe prevista usa o limiar **congelado** de cada modelo, escolhido nas previsões fora do fold (máximo do F1 macro); nunca é afinado no teste.

In [ ]:
cls = FinalEvaluation("classification", train_cls, test_cls, frozen=frozen["classification"])
cls.predict()
cls.timing

### 4.2 Tabela de comparação: validação cruzada e teste

As métricas pedidas pelo enunciado, com a média ± desvio-padrão nos 3 folds anuais e o valor no teste (com IC 95% por bootstrap). A segunda tabela tem todas as métricas no teste (*precision*, *recall* e F1 em macro e micro); na classificação binária, o F1 micro é igual à *accuracy*.

**Como ler:** o F1 macro e a AUC são as métricas principais (a *accuracy* é enganadora com ~15% de *Default*: a classe maioritária tem *accuracy* alta e não apanha nenhum). A taxa de *Default* do teste não é a do treino (secção 5), e o F1 depende dessa taxa; a AUC não, por isso é a melhor comparação direta entre a validação e o teste.

In [ ]:
cls_table = cls.metrics_table(N_BOOT)
cls_table.to_csv(RESULTS_DIR / "classification_test.csv")
cls_table

In [ ]:
cls_all = cls.test_metrics().round(3)
cls_all.to_csv(RESULTS_DIR / "classification_test_all_metrics.csv")
cls_all

### 4.3 O modelo final contra os outros (bootstrap emparelhado)

F1 macro e AUC do modelo final menos os do outro modelo, nas mesmas reamostragens. Positivo quer dizer que o modelo final é melhor. No 04c, a regressão logística empatou com a MLP; aqui vê-se se o empate se mantém em 2015.

In [ ]:
cls_paired = cls.paired_table(n_boot=N_BOOT)
cls_paired.to_csv(RESULTS_DIR / "classification_test_paired.csv")
cls_paired

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 0.45 * len(cls.models) + 1.4), sharey=True)
cls.plot_cv_vs_test("f1_macro", ax=axes[0])
cls.plot_cv_vs_test("auc", ax=axes[1])
axes[1].get_legend().remove()
fig.suptitle("Classificação: validação cruzada e teste de 2015")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "final_cls_cv_teste.png")
plt.show()

### 4.4 Matrizes de confusão do modelo final

A matriz 2 × 2 é a da classificação binária. A 3 × 2 separa as linhas pelas 3 classes do enunciado: mostra se os *At risk* (que no treino não existiam) caem do lado do *Default*, como a decisão binária pressupõe. As percentagens são de cada linha (a linha *Default* dá o *recall* de *Default*).

In [ ]:
confusions = {"2 x 2": cls.confusion(), "2 x 2 (%)": cls.confusion(percent=True),
              "3 x 2": cls.confusion(three_classes=True), "3 x 2 (%)": cls.confusion(three_classes=True, percent=True)}
pd.concat(confusions).to_csv(RESULTS_DIR / "classification_test_confusion.csv")
for name, table in confusions.items():
    print(f"\n{name} ({cls.final_model}, limiar {cls.threshold(cls.final_model)})")
    print(table.to_string())

### 4.5 Mudança temporal: o risco ao longo de 2015

A taxa real de *Default* por mês de emissão, a percentagem que o modelo final classifica como *Default* e a P(Default) média. Se a P(Default) média ficar sempre abaixo da taxa real, o modelo subestima o risco de 2015 (os anos de treino eram menos arriscados, como a EDA mostrou), mesmo que ordene bem os empréstimos (AUC).

In [ ]:
cls_months = cls.by_period()
axes = cls.plot_by_period()
axes[0].figure.suptitle(f"Classificação ({cls.final_model}): Default por mês de emissão em 2015")
axes[0].figure.tight_layout()
axes[0].figure.savefig(config.FIGURES_DIR / "final_cls_por_mes.png")
plt.show()
cls_months

### 4.6 Equidade: recall e falsos alarmes por grupo

Para cada grupo: a taxa real de *Default*, o *recall* de *Default* (defaults que o modelo apanha) e os falsos alarmes (bons empréstimos classificados como *Default*, que na prática seriam recusados ou mais caros). Diferenças grandes de falsos alarmes entre grupos com a mesma taxa real seriam um problema de equidade; diferenças que acompanham a taxa real são o modelo a fazer o seu trabalho, e convém separar as duas coisas na discussão.

In [ ]:
cls_groups = pd.concat({by: cls.subgroup_table(by) for by in GROUPS})
cls_groups.to_csv(RESULTS_DIR / "classification_test_subgroups.csv")
fig = cls.plot_subgroups(GROUPS, ["Default real (%)", "recall Default", "falsos alarmes (%)"])
fig.savefig(config.FIGURES_DIR / "final_cls_equidade.png")
plt.show()
cls_groups

## 5. Mudança temporal: as variáveis no treino e no teste

Só agora, com os modelos congelados, se comparam as distribuições do treino e do teste. O PSI (*population stability index*) resume a mudança de cada variável: abaixo de 0,1 é estável, de 0,1 a 0,25 é uma mudança moderada e acima de 0,25 é uma mudança grande. Na classificação a distância é grande por construção (treino de 2007–2011, teste de 2015); na regressão o treino acaba em 2014.

In [ ]:
drift = pd.concat({
    "regressão": reg.drift_table(["int_rate", "fico_range_low", "dti", "annual_inc", "loan_amnt",
                                  "revol_util", "emp_length", "term", "home_ownership", "purpose"]),
    "classificação": cls.drift_table([config.CLASS_COL, "int_rate", "grade", "fico_range_low", "dti",
                                      "annual_inc", "loan_amnt", "revol_util", "home_ownership", "purpose"]),
})
drift.to_csv(RESULTS_DIR / "test_drift.csv")
drift

## 6. Resumo para o relatório e para o sumário executivo

Os números principais, calculados a partir das tabelas acima.

In [ ]:
# Regressão
r, test_r = reg.final_model, reg.test_metrics()
mean, std = parse_mean_std(reg.entries[r]["cv"].get("mae"))
mae = test_r.loc[r, "mae"]
print(f"REGRESSÃO - {r}: MAE no teste {mae:.3f} p.p. {reg_table.loc[r, 'IC 95% MAE']} "
      f"(CV {mean:.3f} ± {std:.3f}: {'dentro' if abs(mae - mean) <= std else 'fora'} de um desvio)")
for base in ["Mediana por prazo", "Mediana"]:
    if base in test_r.index:
        print(f"  melhoria face à {base.lower()}: {100 * (1 - mae / test_r.loc[base, 'mae']):.1f}%")
print(f"  R² no teste {test_r.loc[r, 'r2']:.3f}; viés médio {reg_months['viés'].mean():+.3f} p.p. "
      f"(meses de {reg_months['viés'].min():+.3f} a {reg_months['viés'].max():+.3f})")

# Classificação
c, test_c = cls.final_model, cls.test_metrics()
conf = cls.confusion()
caught = conf.loc["Default", "Default"] / conf.loc["Default"].sum()
alarms = conf.loc["Good standing", "Default"] / conf.loc["Good standing"].sum()
three = cls.confusion(three_classes=True)
print(f"\nCLASSIFICAÇÃO - {c} (limiar {cls.threshold(c)}): F1 macro {test_c.loc[c, 'f1_macro']:.3f} "
      f"{cls_table.loc[c, 'IC 95% F1 macro']}, AUC {test_c.loc[c, 'auc']:.3f} {cls_table.loc[c, 'IC 95% AUC']}")
print(f"  em cada 100 defaults de 2015, o modelo apanha {100 * caught:.0f}; "
      f"em cada 100 bons empréstimos, marca {100 * alarms:.0f} como Default")
if three.loc["At risk"].sum():
    print(f"  At risk classificados como Default: {100 * three.loc['At risk', 'Default'] / three.loc['At risk'].sum():.0f}% "
          f"({three.loc['At risk'].sum():,} empréstimos)")
print(f"  Default real no teste {100 * (cls.y == 'Default').mean():.1f}% "
      f"(no treino {100 * (cls.train[config.CLASS_COL] == 'Default').mean():.1f}%); "
      f"P(Default) média prevista {100 * cls.predict()[c].mean():.1f}%")

# Equidade e deriva
for name, groups, column in [("regressão", reg_groups, "viés"), ("classificação", cls_groups, "falsos alarmes (%)")]:
    for by in GROUPS:
        part = groups.loc[by].drop(index=["total", "outros"], errors="ignore")   # "outros": grupos pequenos
        print(f"\n{name}, {by}: {column} de {part[column].min():.2f} ({part[column].idxmin()}) "
              f"a {part[column].max():.2f} ({part[column].idxmax()})", end="")
print()
for name in ["regressão", "classificação"]:
    top = drift.loc[name].head(3)
    print(f"\nMaior mudança ({name}): " + ", ".join(f"{col} (PSI {psi:.2f})" for col, psi in top["PSI"].items()))

**O que discutir no relatório** (secção de avaliação, com os quatro temas que o enunciado obriga):

1. **Generalização.** O valor no teste contra a média ± desvio da validação cruzada (3.2 e 4.2). Se o modelo final continua à frente das *baselines* e se a diferença para o 2.º se mantém (3.3 e 4.3).
2. **Mudança temporal.** O erro e o viés ao longo de 2015 (3.4 e 4.5) e as variáveis que mais mudaram (secção 5). Na classificação, a distância entre o treino (2007–2011) e o teste (2015) é a maior limitação do desenho; ligar às recomendações de retreino do plano de deployment.
3. **Follow-up desigual.** Resolvido pelo desenho: só 36 meses e só com o prazo terminado. Os números da secção 2 (60 meses e `Current` removidos) e a tabela 3 × 2 (4.4) mostram o custo e onde caem os *At risk*.
4. **Viés de seleção.** Os modelos só viram pedidos aceites; os rejeitados têm um *risk score* muito mais baixo (testes de hipóteses da Parte 1). Os resultados do teste valem para a população que a LendingClub aprovou em 2015, não para todos os pedidos.
5. **Equidade.** As diferenças entre grupos (3.5 e 4.6), separando o que acompanha a taxa real de *Default* do que não acompanha. A geografia ficou fora dos modelos.
6. **Se algum resultado for mau:** reportar e explicar; os modelos não mudam depois de ver o teste.